# **Mapping short-let pressure in Bristol: A neighbourhood analysis of Airbnb listings**

# ML objectives 

# Change working directory

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Users\\chheb\\OneDrive\\Documents\\Programming\\Mapping_Short_Let-Pressure_in_Bristol\\jupyter_notebooks'

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


In [3]:
current_dir = os.getcwd()
current_dir

'c:\\Users\\chheb\\OneDrive\\Documents\\Programming\\Mapping_Short_Let-Pressure_in_Bristol'

# Import transformed data

In [ ]:
from pathlib import Path  
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)


RANDOM_STATE = 42


In [5]:
df = pd.read_csv("data/processed/listings_clean.csv", dtype={"id": str},
                 parse_dates=["first_review", "last_review"])
active = df[df["is_active"]].copy()
print(df.shape, active.shape) 

(2932, 51) (2561, 51)


# 1. Split the data

In [6]:

target = "is_operator"
features = ["accommodates", "price", "is_entire_home", "availability_365",
            "review_scores_rating", "number_of_reviews", "host_is_superhost"]

X = active[features].astype(float)
y = active[target].astype(int)
groups = active["host_key"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(X_train.shape, X_test.shape)
print("Operator share  train:", round(y_train.mean(), 3), " test:", round(y_test.mean(), 3))
print("Hosts in both sets:", len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))   # expect 0
print(X_train.isna().sum())

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("Baseline accuracy on test:", round(baseline.score(X_test, y_test), 3))

(1971, 7) (590, 7)
Operator share  train: 0.397  test: 0.495
Hosts in both sets: 0
accommodates              0
price                   240
is_entire_home            0
availability_365          0
review_scores_rating    179
number_of_reviews         0
host_is_superhost         0
dtype: int64
Baseline accuracy on test: 0.505


In [7]:
top_host = active["host_key"].value_counts().index[0]
print("Largest host in test set:", top_host in set(groups.iloc[test_idx]))
print("Their listings in test:", (groups.iloc[test_idx] == top_host).sum(), "of", len(test_idx))

Largest host in test set: False
Their listings in test: 0 of 590


In [8]:
test_hosts = groups.iloc[test_idx]
print("Hosts in test set:", test_hosts.nunique())
print(test_hosts.value_counts().head(5))

Hosts in test set: 295
host_key
285    65
564    33
534    25
670    21
787    19
Name: count, dtype: int64
